### Introduction

"Automatic Speech Recognition (ASR) is AI technology that converts spoken language into written text by analyzing audio waveforms and identifying patterns in human speech." (Foster, 2026)

It's easy to take this kind of technology for granted because we interact with systems built on top this in our every day lives. For instances, the AI assistants (eg Siri and Google Assistant) used in modern smartphones utilize ASR to understand user speech. These assistants, however, utilize their cloud servers to process our prompts rather tben doing that on device. My goal, in this notebook, is to explore the performance of the various open source ASR models, identify their strengths & weaknesses and see how they compare agaist clould based ASR models.

Below is the set of imports that we will use in this notebook

In [4]:
!pip install -U openai-whisper moonshine-voice

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.9 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 48.4 MB/s eta 0:00:0000:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 6.9 MB/s eta 0:00:00:00:0100:01
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=2238a5bf31635f79ac3f47b7fb0ebcffd7baa061b9a40752af9be0b00a9ac8a9
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [24]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.4 MB/s eta 0:00:0000:0100:01


In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import librosa
from moonshine_voice import (
    Transcriber,
    get_model_for_language,
    ModelArch
)
from typing import List
import jiwer

In [9]:
# Downloads the model files locally and returns the strict system paths
!moonshine-voice download --stt --language en --model-arch 0

encoder_model.ort: 100%|███████████████████| 12.7M/12.7M [00:00<00:00, 60.5MB/s]
decoder_model_merged.ort: 100%|█████████████| 29.0M/29.0M [00:00<00:00, 128MB/s]
tokenizer.bin: 100%|█████████████████████████| 244k/244k [00:00<00:00, 4.61MB/s]
Model download url: https://download.moonshine.ai/model/tiny-en/quantized/tiny-en
Model components: ['encoder_model.ort', 'decoder_model_merged.ort', 'tokenizer.bin']
Model arch: 0
Downloaded model path: /root/.cache/moonshine_voice/download.moonshine.ai/model/tiny-en/quantized/tiny-en


## Dataset


### About the Dataset
We will use the Mozzila common voice dataset for this evaluation. This was obtained from the dataset uploaded by the Mozilla and Rachel Tatman kaggle accounts (https://www.kaggle.com/datasets/mozillaorg/common-voice/data). 

As explained in the README file, the CSV files in the dataset contain metadata about each audio file. This includes the file name, the supposed transciption, the age of the speaker (if reported), the gender of the speaker, the accent of the speaker (the region of the world the person sounds like they are from). The actual audio files are contained in the subdirectories.

### Data Loading

For this section, we will load the dev dataset which we will use to evaluate the different models.

In [2]:
root = "/kaggle/input/datasets/mozillaorg/common-voice"

# we start by loading the dev CSV file.
metadata = pd.read_csv(f"{root}/cv-valid-dev.csv")

print(f"Number of elements: {metadata.size}")

metadata.head()

Number of elements: 32608


,filename,text,up_votes,down_votes,age,gender,accent,duration
0,cv-valid-dev/sample-000000.mp3,be careful with your prognostications said the...,1,0,NaN,NaN,NaN,NaN
1,cv-valid-dev/sample-000001.mp3,then why should they be surprised when they se...,2,0,NaN,NaN,NaN,NaN
2,cv-valid-dev/sample-000002.mp3,a young arab also loaded down with baggage ent...,2,0,NaN,NaN,NaN,NaN
3,cv-valid-dev/sample-000003.mp3,i thought that everything i owned would be des...,3,0,NaN,NaN,NaN,NaN
4,cv-valid-dev/sample-000004.mp3,he moved about invisible but everyone could he...,1,0,fourties,female,england,NaN


Since we are just evaluating model performance, our interest will lie in the file name (so that we load the actual data) and the transcription. The filename is our input and the text transcription is our output.

In [3]:
Y = []
X = []
# now we load the media files
for row in metadata.itertuples():
    with open(f"{root}/cv-valid-dev/{row.filename}") as file:
        X.append(file)
        Y.append(row.text)

print(f"X length: {len(X)}, Y length: {len(Y)} ")

X length: 4076, Y length: 4076 


Now that we have loaded the data, I believe we can now proceed with evaluating the models

## Model evaluation

### Evalutaion Methods and Configs

We will start by setting up what we will use for calculating the Word Error Rate (WER) for the models.

In [41]:
transformation = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfWords()
])

### Moonshine Tiny

We will start by evaluating the performace of the moonshine tiny model. The Moonshine Tiny model is a 26M parameter model with a 12% WER rate. This figure will be crossed checked in our evaluations.

In [29]:
wer_mst_list = [] # list of WERs for each audio file

In [31]:
# this mathod was inspired by the moonshine basic_transcription.py example 
# found at https://github.com/moonshine-ai/moonshine/blob/main/examples/python/basic_transcription.py
def transcribe_without_streaming(
    transcriber: Transcriber, audio_data: List[float], sample_rate: int
):
    """Transcribe audio data offline without streaming."""
    transcript = transcriber.transcribe_without_streaming(
        audio_data, sample_rate=sample_rate, flags=0
    )
    sentences = []
    for line in transcript.lines:
        sentences.append(line.text)

    return " ".join(sentences)

In [18]:
# the model path user was obtained from the "!moonshine-voice download --stt --language en --model-arch 0" install command 
transcriber = Transcriber(model_path="/root/.cache/moonshine_voice/download.moonshine.ai/model/tiny-en/quantized/tiny-en", model_arch=ModelArch.TINY)

In [43]:
for i in range(0, len(X)):
    audio_array, sample_rate = librosa.load(X[i].name, sr=16000, mono=True)
    
    y = transcribe_without_streaming(transcriber, audio_array, sample_rate)
    
    wer = jiwer.wer(
    Y[i], 
    y, 
    reference_transform=transformation, 
    hypothesis_transform=transformation
    )

    wer_mst_list.append(wer)

In [44]:
avg_wer_mst = sum(wer_mst_list)/len(wer_mst_list)
print(f"The average WER is {avg_wer_mst}")

The average WER is 0.20195351192038521
